# 01 — **Data Cleaning & Preparation**

**Dataset:** [UK E-Commerce Data](https://www.kaggle.com/datasets/carrie1/ecommerce-data) (Kaggle: `carrie1/ecommerce-data`)
**Author:** Ayush, Data Analyst
**Last updated:** 2026-09-05

## Overview

This notebook loads the raw transactional dataset, diagnoses data quality issues, and produces a clean, analysis-ready dataset for downstream notebooks:

- RFM segmentation
- Cohort analysis
- Revenue trend analysis

## 1. Load Raw Data

Importing the required Python libraries and load the dataset into a pandas DataFrame. The data will then be examined and prepared for further analysis and will be encoded accordingly 

In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
import seaborn as sns 
from charset_normalizer import from_path
import warnings 
warnings.filterwarnings('ignore')

In [2]:
result = from_path(r'D:\ProProjects\data-analytics-projects\ecom\dataset\data.csv').best()
print(result.encoding)

cp1250


In [3]:
df_raw = pd.read_csv(r'D:\ProProjects\data-analytics-projects\ecom\dataset\data.csv' , encoding = 'cp1250')
df_raw.sample()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
507919,579187,85176,SEWING SUSAN 21 NEEDLE SET,1,11/28/2011 15:31,1.63,NaN,United Kingdom


In [4]:
df_raw.columns = df_raw.columns.astype(str).str.lower()
df_raw.columns

Index(['invoiceno', 'stockcode', 'description', 'quantity', 'invoicedate',
       'unitprice', 'customerid', 'country'],
      dtype='object')

In [5]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   invoiceno    541909 non-null  object 
 1   stockcode    541909 non-null  object 
 2   description  540455 non-null  object 
 3   quantity     541909 non-null  int64  
 4   invoicedate  541909 non-null  object 
 5   unitprice    541909 non-null  float64
 6   customerid   406829 non-null  float64
 7   country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


## 2. Initial data quality audit

Quantifying every issue so cleaning decisions are evidence-based rather than assumption based

In [8]:
audit = {}

audit['total_rows'] = len(df_raw)
audit['missing_customerid'] = df_raw['customerid'].isna().sum()
audit['missing_description'] = df_raw['description'].isna().sum()
audit['negative_or_zero_quantity'] = (df_raw['quantity'] <= 0).sum()
audit['negative_or_zero_unitPprice'] = (df_raw['unitprice'] <= 0).sum()
audit['cancelled_invoices'] = df_raw['invoiceno'].astype(str).str.startswith('C').sum()
audit['duplicate_rows'] = df_raw.duplicated().sum()
audit['unique_customers'] = df_raw['customerid'].nunique()
audit['unique_products'] = df_raw['stockcode'].nunique()
audit['unique_countries'] = df_raw['country'].nunique()
audit['unique_invoices'] = df_raw['invoiceno'].nunique()

pd.Series(audit)

total_rows                     541909
missing_customerid             135080
missing_description              1454
negative_or_zero_quantity       10624
negative_or_zero_unitPprice      2517
cancelled_invoices               9288
duplicate_rows                   5268
unique_customers                 4372
unique_products                  4070
unique_countries                   38
unique_invoices                 25900
dtype: int64

In [9]:
# Checking how a cancellation actually look like?
df_raw[df_raw['invoiceno'].astype(str).str.startswith('C')].head(3)

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
141,C536379,D,Discount,-1,12/1/2010 9:41,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,12/1/2010 9:49,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,12/1/2010 10:24,1.65,17548.0,United Kingdom


In [11]:
# Checking how do rows with missing CustomerID look like? Are they valid transactions?
df_raw[df_raw['customerid'].isna()].head(3)

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
622,536414,22139,NaN,56,12/1/2010 11:52,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,12/1/2010 14:32,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,12/1/2010 14:32,2.51,NaN,United Kingdom


In [12]:
# Check for non-numeric stockcodes that may represent non-product entries.
odd_codes = df_raw[~df_raw['stockcode'].astype(str).str.contains(r'^\d')]['stockcode'].value_counts()
odd_codes.head(15)

stockcode
POST            1256
DOT              710
M                571
C2               144
D                 77
S                 63
BANK CHARGES      37
AMAZONFEE         34
CRUK              16
DCGSSGIRL         13
DCGSSBOY          11
gift_0001_20      10
gift_0001_10       9
gift_0001_30       8
DCGS0003           5
Name: count, dtype: int64